# Experiment 6b-v2: S5-Relay Syncretism — Validation Ladder

This notebook does **not** run the full eps×d×seed sweep from the redesign
spec. It runs the cheap-to-expensive validation ladder first, in order, so a
broken mechanism or instrument is caught before real sweep compute is spent:

1. **No-training sanity checks** — preamble byte-identity across L_A/L_B/L_C,
   a count-bucket shortcut-resistance audit on the *actual* `compose_chain`
   implementation (not just the abstract claim), and `f_readout` class
   balance.
2. **Coarse L_A-only calibration gate** (spec §2, blocking) — a small grid
   over `m` and `d`, one seed, tracking accuracy **over training steps** (not
   just final accuracy) to distinguish "climbing, just needs more steps" from
   "flat at chance, genuinely too hard" — that qualitative shape is the whole
   point of this gate.
3. **One instrumented pilot point** — after picking a calibrated `m` from
   step 2, train a *single* shared syncretism checkpoint (one `d`, one seed)
   and run both the terminal-recovery probe (targets `f(G)` at the
   syncretized readout leaf) and the IoI-analog control probe (targets the
   *visible* relay state — should show no cross-language gap) on it. This is
   the step that was skipped in the original Exp 6b run, where the first
   real signal check happened only after a full 24-checkpoint sweep.

If step 3 looks right — L_A recovers `f(G)` near ceiling, L_B/L_C show a real
gap, and the control probe is flat — that's the green light for the real
sweep (spec §3/§4: disjoint-grammar control with the same relay burden,
budget-matched dedicated models, A′ calibration). None of those are built
here; see the wrap-up cell at the end.

Reuses, verbatim where possible, the grammar/model/training infra from the
attached `Experiment6b_Syncretism_BudgetMatched.ipynb` (the currently-running
terminal-collapse design) so results stay comparable and the plumbing is
already trust-worthy.

## 0. Device setup

In [1]:
import os
import torch
DEVICE = torch.device("cuda:5" if torch.cuda.is_available() else "cpu")
print(f"using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  {torch.cuda.get_device_name(2)}")

from contextlib import nullcontext
def autocast_ctx():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16) if DEVICE.type == "cuda" else nullcontext()

using device: cuda:5
  NVIDIA H200 NVL


## 1. Config

Grammar shape, word-order design, mix, and the syncretism surface maps are
copied verbatim from Exp 6b so this stays comparable to the existing runs.
**New:** `GEN_S`/`GEN_C` vocabulary tokens for the S5-relay preamble, appended
after the BOS tags.

In [2]:
# ============================== CONFIG (unchanged from Exp 6b) ==============
LEVELS = 6
NT_SIZE = 4
TERM_SIZE = 4
MIN_DEGREE = 2
MAX_DEGREE = 4
RULE_LEN_CHOICES = [2, 3]

LANGS = {
    "L_A": set(),
    "L_B": {4, 5},
    "L_C": set(range(1, LEVELS + 1)),
}
LANG_NAMES = list(LANGS.keys())
MIX = {"L_A": 0.70, "L_B": 0.15, "L_C": 0.15}

def round_to_valid_n_embd(x, n_head):
    unit = 2 * n_head
    return max(unit, round(x / unit) * unit)

# --------------------------- SYNCRETISM (unchanged from Exp 6b) -------------
EPSILON_LEVELS = {
    0: {},
    1: {3: 2},
    2: {2: 1, 3: 1},
}

def make_surface_map(eps_level):
    override = EPSILON_LEVELS[eps_level]
    m = {t: t for t in range(TERM_SIZE)}
    m.update(override)
    return m

# --------------------------- NEW: S5-RELAY VOCAB -----------------------------
# Vocab layout: [0 .. TERM_SIZE-1] terminals, [TERM_SIZE .. TERM_SIZE+2] BOS
# tags (one per language), then GEN_S, GEN_C as two more atomic tokens.
BOS_OFFSET = TERM_SIZE
BOS = {name: BOS_OFFSET + i for i, name in enumerate(LANG_NAMES)}
GEN_S = BOS_OFFSET + len(LANG_NAMES)
GEN_C = BOS_OFFSET + len(LANG_NAMES) + 1
VOCAB_SIZE = GEN_C + 1

print("surface maps:")
for k in EPSILON_LEVELS:
    print(f"  eps_level={k}: {make_surface_map(k)}")
print(f"BOS tags: {BOS}")
print(f"GEN_S={GEN_S}, GEN_C={GEN_C}, VOCAB_SIZE={VOCAB_SIZE}")

surface maps:
  eps_level=0: {0: 0, 1: 1, 2: 2, 3: 3}
  eps_level=1: {0: 0, 1: 1, 2: 2, 3: 2}
  eps_level=2: {0: 0, 1: 1, 2: 1, 3: 1}
BOS tags: {'L_A': 4, 'L_B': 5, 'L_C': 6}
GEN_S=7, GEN_C=8, VOCAB_SIZE=9


## 2. Grammar builder, sampler, linearizer (unchanged from Exp 6b)

`sample_tree_indexed` / `linearize_with_paths` are the instrumented versions
(instance ids on internal NT nodes, canonical leaf ids assigned once at
sampling time, independent of word order) — needed here so the "target
terminal chosen by tree-node identity, not surface position" requirement
from the spec is trivial to satisfy: we designate the leaf with
**canonical `leaf_id == 0`** (the first leaf sampled, depth-first, before any
language's word-order reversal is applied) as the readout leaf.

In [3]:
import random
from collections import defaultdict

def build_grammar(seed):
    rng = random.Random(seed)
    symbols_by_level = {1: [(1, 0)]}
    for l in range(2, LEVELS + 1):
        symbols_by_level[l] = [(l, i) for i in range(NT_SIZE)]
    terminals = list(range(TERM_SIZE))

    rules = defaultdict(list)
    for l in range(1, LEVELS + 1):
        child_level = l + 1
        child_symbols = terminals if child_level == LEVELS + 1 else symbols_by_level[child_level]
        for sym in symbols_by_level[l]:
            degree = rng.randint(MIN_DEGREE, MAX_DEGREE)
            seen = set()
            tries = 0
            while len(rules[sym]) < degree and tries < 200:
                tries += 1
                rule_len = rng.choice(RULE_LEN_CHOICES)
                rhs = tuple(rng.choice(child_symbols) for _ in range(rule_len))
                if rhs in seen:
                    continue
                seen.add(rhs)
                rules[sym].append(rhs)
    return rules, symbols_by_level


def sample_tree_indexed(rules, rng, symbol=(1, 0), level=1, _id_counter=None, _inst_counter=None):
    if _id_counter is None: _id_counter = [0]
    if _inst_counter is None: _inst_counter = [0]
    inst = _inst_counter[0]; _inst_counter[0] += 1
    rhs = rng.choice(rules[symbol])
    children = []
    for child_sym in rhs:
        if level == LEVELS:
            children.append(("LEAF", child_sym, _id_counter[0])); _id_counter[0] += 1
        else:
            children.append(sample_tree_indexed(rules, rng, child_sym, level + 1, _id_counter, _inst_counter))
    return (symbol, children, inst)


def linearize_with_paths(node, reverse_levels, _path=()):
    # Returns list of (true_terminal, canonical_leaf_id, nt_path) per leaf,
    # in THIS language's linearization order. leaf_id is assigned once at
    # sampling time (canonical, pre-order, word-order-independent) -- this is
    # what lets us pick a target leaf "by tree-node identity, not surface
    # position" per the spec's blocking requirement.
    if isinstance(node, tuple) and len(node) == 3 and node[0] == "LEAF":
        _, tok, leaf_id = node
        return [(tok, leaf_id, _path)]
    (level, idx), children, inst = node
    order = list(reversed(children)) if level in reverse_levels else children
    new_path = _path + ((level, idx, inst),)
    out = []
    for c in order:
        out.extend(linearize_with_paths(c, reverse_levels, new_path))
    return out


def n_leaves(tree):
    if isinstance(tree, tuple) and len(tree) == 3 and tree[0] == "LEAF":
        return 1
    return sum(n_leaves(c) for c in tree[1])


# smoke test
check_rules, check_symbols_by_level = build_grammar(seed=0)
_t = sample_tree_indexed(check_rules, random.Random(1))
_a = linearize_with_paths(_t, LANGS["L_A"])
_c = linearize_with_paths(_t, LANGS["L_C"])
print(f"sample tree yields {len(_a)} leaves under L_A")
print("leaf_ids are word-order-independent (same MULTISET under A and C)?",
      sorted(l for _, l, _ in _a) == sorted(l for _, l, _ in _c))

sample tree yields 225 leaves under L_A
leaf_ids are word-order-independent (same MULTISET under A and C)? True


## 3. Model — TinyRotaryGPT (unchanged from Exp 6b)

In [4]:
import torch.nn as nn
import torch.nn.functional as F

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert (n_embd // n_head) % 2 == 0, "head_dim must be EVEN for rotary embeddings"
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

# smoke test
torch.manual_seed(0)
_m = TinyRotaryGPT(VOCAB_SIZE, n_embd=8, n_layer=2, n_head=2, max_seq_len=64).to(DEVICE)
_idx = torch.randint(0, VOCAB_SIZE, (4, 32), device=DEVICE)
_logits, _ = _m(_idx, _idx)
print(f"forward pass OK on {DEVICE}, output shape:", _logits.shape)

forward pass OK on cuda:5, output shape: torch.Size([4, 32, 9])


## 4. Global run constants (same grammar seed as Exp 6/6b)

In [5]:
check_rules, check_symbols_by_level = build_grammar(seed=0)
N_HEAD = 4
SEQ_LEN_CHECK = 256
FIXED_BUDGET_SWEEP = 90_000     # real sweep's step budget, for reference -- not used at full length here
MAX_PAR = 20

print(f"grammar built (seed=0): {sum(len(v) for v in check_rules.values())} total rules across "
      f"{len(check_rules)} symbols")

grammar built (seed=0): 65 total rules across 21 symbols


## 5. The S5-relay mechanism

`s = (1,0,2,3,4)` (transposition), `c = (1,2,3,4,0)` (5-cycle) — generators
of S5, exactly as specified. `compose(p, q)`: apply `p` then `q`, i.e.
`compose(p,q)[i] = q[p[i]]`. `compose_chain` folds left-to-right, matching
`G = g_1 · g_2 · ... · g_m` applied in that order.

In [6]:
GEN_PERM = {0: (1, 0, 2, 3, 4), 1: (1, 2, 3, 4, 0)}   # bit 0 -> s, bit 1 -> c
GEN_TOKEN = {0: GEN_S, 1: GEN_C}
IDENTITY_PERM = (0, 1, 2, 3, 4)

def compose(p, q):
    # Apply p then q: compose(p,q)[i] = q[p[i]].
    return tuple(q[p[i]] for i in range(5))

def sample_relay_chain(m, rng):
    # m iid coin flips choosing generator s (0) or c (1).
    return [rng.randint(0, 1) for _ in range(m)]

def compose_chain(bits):
    # G = g_1 . g_2 . ... . g_m, left to right.
    G = IDENTITY_PERM
    for b in bits:
        G = compose(G, GEN_PERM[b])
    return G

def f_readout(G):
    # Relational, balanced readout (NOT G[0]==0, which is ~20/80).
    return 1 if G[0] < G[1] else 0

def relay_preamble_tokens(bits):
    return [GEN_TOKEN[b] for b in bits]

# smoke test
_rng = random.Random(0)
_bits = sample_relay_chain(8, _rng)
_G = compose_chain(_bits)
print(f"bits={_bits}")
print(f"G={_G}  f(G)={f_readout(_G)}")
print(f"preamble tokens={relay_preamble_tokens(_bits)}")

bits=[1, 1, 0, 1, 1, 1, 1, 1]
G=(2, 3, 4, 1, 0)  f(G)=1
preamble tokens=[8, 8, 7, 8, 8, 8, 8, 8]


## 6. Step 1a — count-bucket shortcut-resistance audit (NO training)

The design's central empirical claim: `f_readout(compose_chain(bits))` is
**not** determined by the count of one generator alone (a count-based
mechanism is exactly what a single attention head can shortcut). This was
confirmed once, abstractly — rerun it here against the *actual*
`compose_chain`/`GEN_PERM` code above, since a convention bug (composition
order, generator aliasing) would silently produce a solvable-looking or
trivial group and this notebook would be testing the wrong thing.

Contrast: a **parity** readout (`count(bit==1) % 2`) is *fully* reducible to
count by construction — every count-bucket is pure for it. If the S5 readout
comes back similarly pure (most buckets constant), the generators aren't
doing their job and `compose_chain`/`GEN_PERM` need to be checked for bugs
before anything else in this notebook is trusted.

In [7]:
def count_bucket_audit(m=32, n_samples=200_000, min_bucket_n=30, seed=0):
    rng = random.Random(seed)
    buckets_s5 = defaultdict(set)
    buckets_parity_input = defaultdict(list)   # count -> list of parity labels (sanity: always pure)
    counts = defaultdict(int)
    for _ in range(n_samples):
        bits = [rng.randint(0, 1) for _ in range(m)]
        cnt = sum(bits)
        label_s5 = f_readout(compose_chain(bits))
        label_parity = cnt % 2
        buckets_s5[cnt].add(label_s5)
        buckets_parity_input[cnt].append(label_parity)
        counts[cnt] += 1

    testable = [cnt for cnt, n in counts.items() if n >= min_bucket_n]
    mixed_s5 = sum(1 for cnt in testable if len(buckets_s5[cnt]) > 1)
    mixed_parity = sum(1 for cnt in testable if len(set(buckets_parity_input[cnt])) > 1)

    print(f"m={m}, {n_samples} sampled chains, {len(testable)} count-buckets with >= {min_bucket_n} samples")
    print(f"  S5 readout f(G):      {mixed_s5}/{len(testable)} buckets MIXED (both outcomes present)")
    print(f"  parity(count) 'shortcut': {mixed_parity}/{len(testable)} buckets mixed (should be 0 -- fully "
          f"count-determined by construction; this is the sanity baseline, not a claim about the real mechanism)")
    if mixed_s5 < 0.7 * len(testable):
        print("  *** WARNING: S5 readout looks unexpectedly count-predictable -- check compose()/GEN_PERM "
              "for a convention bug before trusting anything downstream. ***")
    else:
        print("  Looks right: count alone does not determine the label for the large majority of buckets.")
    return mixed_s5, len(testable), mixed_parity

_ = count_bucket_audit(m=32)
print()
_ = count_bucket_audit(m=16)

m=32, 200000 sampled chains, 21 count-buckets with >= 30 samples
  S5 readout f(G):      21/21 buckets MIXED (both outcomes present)
  parity(count) 'shortcut': 0/21 buckets mixed (should be 0 -- fully count-determined by construction; this is the sanity baseline, not a claim about the real mechanism)
  Looks right: count alone does not determine the label for the large majority of buckets.

m=16, 200000 sampled chains, 15 count-buckets with >= 30 samples
  S5 readout f(G):      15/15 buckets MIXED (both outcomes present)
  parity(count) 'shortcut': 0/15 buckets mixed (should be 0 -- fully count-determined by construction; this is the sanity baseline, not a claim about the real mechanism)
  Looks right: count alone does not determine the label for the large majority of buckets.


## 7. Step 1b — `f_readout` class balance (NO training)

In [8]:
def balance_check(m, n_samples=200_000, seed=1):
    rng = random.Random(seed)
    ones = sum(f_readout(compose_chain([rng.randint(0, 1) for _ in range(m)])) for _ in range(n_samples))
    frac = ones / n_samples
    print(f"m={m}: P(f(G)=1) = {frac:.4f}  (expect ~0.50 for a genuinely balanced relational readout)")
    return frac

for _m in (8, 16, 32):
    balance_check(_m)

m=8: P(f(G)=1) = 0.5183  (expect ~0.50 for a genuinely balanced relational readout)
m=16: P(f(G)=1) = 0.5114  (expect ~0.50 for a genuinely balanced relational readout)
m=32: P(f(G)=1) = 0.4997  (expect ~0.50 for a genuinely balanced relational readout)


## 8. Step 1c — preamble byte-identity check (NO training)

**Blocking requirement from the spec.** The m generator tokens must be an
identical, fixed-order preamble immediately after BOS for all three
languages, regardless of each language's `reverse_levels` — otherwise L_C's
full reversal would present the generators in a different order than the one
the label was computed from, silently reintroducing a word-order/ε
confound. Verify directly rather than trusting the implementation by
inspection.

In [9]:
def build_sequence(tree, lang, bits, readout_leaf_id, readout_label, surface_map):
    # [BOS] + preamble (m gen tokens) + linearized tree, with the leaf whose
    # canonical leaf_id == readout_leaf_id overridden to terminal {0,1} =
    # readout_label, then passed through surface_map like every other
    # terminal (so syncretism collapse applies to it exactly like any other
    # leaf when surface_map is non-identity).
    seq = [BOS[lang]] + relay_preamble_tokens(bits)
    for true_t, leaf_id, _path in linearize_with_paths(tree, LANGS[lang]):
        if leaf_id == readout_leaf_id:
            true_t = readout_label   # override: true identity is f(G), not the grammar draw
        seq.append(surface_map[true_t])
    return seq


rng = random.Random(42)
_tree = sample_tree_indexed(check_rules, rng)
_bits = sample_relay_chain(16, rng)
_label = f_readout(compose_chain(_bits))
_surf_identity = make_surface_map(0)

streams = {lang: build_sequence(_tree, lang, _bits, readout_leaf_id=0, readout_label=_label,
                                 surface_map=_surf_identity)
           for lang in LANG_NAMES}

m = len(_bits)
preambles = {lang: streams[lang][1:1 + m] for lang in LANG_NAMES}
print("preamble region (positions 1..m) per language:")
for lang, p in preambles.items():
    print(f"  {lang}: {p}")

all_identical = len(set(tuple(p) for p in preambles.values())) == 1
print(f"\npreamble byte-identical across L_A/L_B/L_C? {all_identical}")
assert all_identical, "BLOCKING: preamble differs across languages -- fix before proceeding"
print("Tree lengths differ across languages only in the POST-preamble region (expected -- that's word order):")
for lang in LANG_NAMES:
    print(f"  {lang}: total length {len(streams[lang])}")

preamble region (positions 1..m) per language:
  L_A: [7, 8, 8, 8, 8, 7, 7, 8, 8, 7, 8, 8, 7, 8, 7, 8]
  L_B: [7, 8, 8, 8, 8, 7, 7, 8, 8, 7, 8, 8, 7, 8, 7, 8]
  L_C: [7, 8, 8, 8, 8, 7, 7, 8, 8, 7, 8, 8, 7, 8, 7, 8]

preamble byte-identical across L_A/L_B/L_C? True
Tree lengths differ across languages only in the POST-preamble region (expected -- that's word order):
  L_A: total length 169
  L_B: total length 169
  L_C: total length 169


## 9. L_A-only calibration dataset (spec §2)

No multilinguality, no syncretism — just the relay preamble + a real PCFG
tree (L_A word order = identity) + the readout leaf, unambiguous. This
isolates exactly the capacity-sensitive computation shared by all three
languages, so we can find the `m` band where it's "learnable but not
trivial" on the real architecture before any B/C comparison is meaningful.

In [10]:
def make_relay_only_dataset(rules, n_trees, seed, seq_len, m, max_leaves=None):
    # Returns (data, labels, positions). Unlike the mixed-tree pretraining
    # stream, calibration needs one tree per row with a KNOWN readout
    # position, so rows are padded/truncated individually instead of being
    # packed end-to-end.
    rng = random.Random(seed)
    surf = make_surface_map(0)   # no syncretism in the calibration task
    rows, labels, positions = [], [], []
    cap = seq_len - 1 - m   # leave room for BOS + preamble
    while len(rows) < n_trees:
        tree = sample_tree_indexed(rules, rng)
        leaves = linearize_with_paths(tree, LANGS["L_A"])
        if len(leaves) > cap or (max_leaves is not None and len(leaves) > max_leaves):
            continue
        bits = sample_relay_chain(m, rng)
        label = f_readout(compose_chain(bits))
        seq = [BOS["L_A"]] + relay_preamble_tokens(bits)
        target_pos = None
        for true_t, leaf_id, _ in leaves:
            if leaf_id == 0:
                true_t = label
                target_pos = len(seq)
            seq.append(surf[true_t])
        assert target_pos is not None
        pad = seq_len + 1 - len(seq)
        seq = seq + [0] * pad          # pad with terminal-0 id; masked out by loss/eval, never scored
        rows.append(seq)
        labels.append(label)
        positions.append(target_pos)
    return (torch.tensor(rows, dtype=torch.long),
            torch.tensor(labels, dtype=torch.long),
            torch.tensor(positions, dtype=torch.long))


# smoke test
_d, _l, _p = make_relay_only_dataset(check_rules, 200, seed=1, seq_len=SEQ_LEN_CHECK, m=8)
print(f"data shape {_d.shape}, label balance {_l.float().mean().item():.3f}, "
      f"target_pos range [{_p.min().item()}, {_p.max().item()}]")

data shape torch.Size([200, 257]), label balance 0.530, target_pos range [9, 9]


## 10. Calibration training loop with accuracy-over-steps tracking

This trains **in-process** (not via the parallel `train_worker.py`
subprocess launcher used for the real sweep) because the whole point of this
gate is the *qualitative shape* of the accuracy curve — climbing vs. flat —
which needs periodic evaluation during training, not just a final number.
Restricted 2-way accuracy at the known readout position, evaluated on
held-out trees.

In [15]:
@torch.no_grad()
def eval_relay_accuracy(model, data, labels, positions, bs=512, device=DEVICE):
    model.eval()
    correct = total = 0
    pred_sum = 0
    for i in range(0, data.shape[0], bs):
        b = data[i:i + bs].to(device)
        pos = positions[i:i + bs].to(device)
        lab = labels[i:i + bs].to(device)
        with autocast_ctx():
            logits, _ = model(b)
        pred_logits = torch.gather(logits, 1, (pos - 1).view(-1, 1, 1).expand(-1, 1, logits.shape[-1])).squeeze(1)
        pred = pred_logits[:, :2].argmax(-1)
        correct += (pred == lab).sum().item()
        pred_sum += pred.sum().item()
        total += lab.shape[0]
    model.train()
    return correct / total, pred_sum / total   # accuracy, mean predicted class (collapse tell)


def train_relay_calibration(d, m, seed, train_data, train_labels, train_pos,
                            eval_data, eval_labels, eval_pos,
                            max_steps=20_000, eval_every=2000, batch=64, lr=3e-3,
                            n_layer=12, n_head=N_HEAD, device=DEVICE):
    torch.manual_seed(seed)
    model = TinyRotaryGPT(VOCAB_SIZE, n_embd=d, n_layer=n_layer, n_head=n_head,
                          max_seq_len=SEQ_LEN_CHECK + 1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    train_data_dev = train_data.to(device)
    n = train_data_dev.shape[0]
    curve = []
    for step in range(max_steps):
        idx = torch.randint(0, n, (batch,), device=device)
        b = train_data_dev[idx]
        x, y = b[:, :-1], b[:, 1:]
        with autocast_ctx():
            _, loss = model(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
        if (step + 1) % eval_every == 0 or step == max_steps - 1:
            acc, pred_mean = eval_relay_accuracy(model, eval_data, eval_labels, eval_pos, device=device)
            curve.append((step + 1, acc, loss.item(), pred_mean))
    return curve


def classify_curve(curve, chance=0.5, solved_thresh=0.85, climb_thresh=0.05):
    # Cheap heuristic classification of the accuracy-vs-steps shape.
    accs = [a for _, a, _ in curve]
    last, first_half_last = accs[-1], accs[len(accs) // 2]
    still_climbing = (last - first_half_last) > climb_thresh
    if last >= solved_thresh:
        return "SOLVED" if not still_climbing else "CLIMBING (near ceiling)"
    if last <= chance + 0.05:
        return "FLAT AT CHANCE (still climbing)" if still_climbing else "FLAT AT CHANCE (genuinely stuck)"
    return "CLIMBING (partial)" if still_climbing else "PLATEAUED (partial, below ceiling)"

## 11. Run the coarse calibration grid

Coarse on purpose: 1 seed, a handful of `(d, m)` points, fewer steps than
the real sweep's budget. This is only meant to locate the cliff band, not to
pin it down precisely — widen the grid or raise `max_steps` around whatever
looks ambiguous before trusting a boundary.

In [12]:
CALIB_D_LIST = [8, 32, 96]
CALIB_M_LIST = [8, 16, 24, 32]
CALIB_SEED = 0
CALIB_MAX_STEPS = 20_000       # raise if curves haven't clearly plateaued
CALIB_EVAL_EVERY = 2000
CALIB_N_TRAIN_TREES = 8_000
CALIB_N_EVAL_TREES = 1_000

calibration_results = {}
_calib_data_cache = {}

for m in CALIB_M_LIST:
    if m not in _calib_data_cache:
        train_d, train_l, train_p = make_relay_only_dataset(check_rules, CALIB_N_TRAIN_TREES,
                                                             seed=100 + m, seq_len=SEQ_LEN_CHECK, m=m)
        eval_d, eval_l, eval_p = make_relay_only_dataset(check_rules, CALIB_N_EVAL_TREES,
                                                         seed=900 + m, seq_len=SEQ_LEN_CHECK, m=m)
        _calib_data_cache[m] = (train_d, train_l, train_p, eval_d, eval_l, eval_p)
    train_d, train_l, train_p, eval_d, eval_l, eval_p = _calib_data_cache[m]

    for d in CALIB_D_LIST:
        assert d % N_HEAD == 0 and (d // N_HEAD) % 2 == 0, f"d={d} invalid for rotary"
        print(f"training d={d}, m={m}, seed={CALIB_SEED} ...")
        curve = train_relay_calibration(d, m, CALIB_SEED, train_d, train_l, train_p,
                                        eval_d, eval_l, eval_p,
                                        max_steps=CALIB_MAX_STEPS, eval_every=CALIB_EVAL_EVERY)
        calibration_results[(d, m)] = curve
        verdict = classify_curve(curve)
        print(f"  d={d:>3d} m={m:>3d}: final acc={curve[-1][1]:.3f}  -> {verdict}")

print("\ncalibration grid done.")

training d=8, m=8, seed=0 ...
  d=  8 m=  8: final acc=0.627  -> CLIMBING (partial)
training d=32, m=8, seed=0 ...
  d= 32 m=  8: final acc=1.000  -> SOLVED
training d=96, m=8, seed=0 ...
  d= 96 m=  8: final acc=1.000  -> SOLVED
training d=8, m=16, seed=0 ...
  d=  8 m= 16: final acc=0.482  -> FLAT AT CHANCE (genuinely stuck)
training d=32, m=16, seed=0 ...
  d= 32 m= 16: final acc=0.498  -> FLAT AT CHANCE (genuinely stuck)
training d=96, m=16, seed=0 ...
  d= 96 m= 16: final acc=0.539  -> FLAT AT CHANCE (genuinely stuck)
training d=8, m=24, seed=0 ...
  d=  8 m= 24: final acc=0.491  -> FLAT AT CHANCE (genuinely stuck)
training d=32, m=24, seed=0 ...
  d= 32 m= 24: final acc=0.491  -> FLAT AT CHANCE (genuinely stuck)
training d=96, m=24, seed=0 ...
  d= 96 m= 24: final acc=0.491  -> FLAT AT CHANCE (genuinely stuck)
training d=8, m=32, seed=0 ...
  d=  8 m= 32: final acc=0.502  -> FLAT AT CHANCE (genuinely stuck)
training d=32, m=32, seed=0 ...
  d= 32 m= 32: final acc=0.499  -> FLAT A

In [13]:
CALIB_D_LIST = [8, 32, 96]
CALIB_M_LIST = [8, 16, 24, 32]
CALIB_SEED = 0
CALIB_MAX_STEPS = 60_000       # raise if curves haven't clearly plateaued
CALIB_EVAL_EVERY = 2000
CALIB_N_TRAIN_TREES = 8_000
CALIB_N_EVAL_TREES = 1_000

calibration_results = {}
_calib_data_cache = {}

for m in CALIB_M_LIST:
    if m not in _calib_data_cache:
        train_d, train_l, train_p = make_relay_only_dataset(check_rules, CALIB_N_TRAIN_TREES,
                                                             seed=100 + m, seq_len=SEQ_LEN_CHECK, m=m)
        eval_d, eval_l, eval_p = make_relay_only_dataset(check_rules, CALIB_N_EVAL_TREES,
                                                         seed=900 + m, seq_len=SEQ_LEN_CHECK, m=m)
        _calib_data_cache[m] = (train_d, train_l, train_p, eval_d, eval_l, eval_p)
    train_d, train_l, train_p, eval_d, eval_l, eval_p = _calib_data_cache[m]

    for d in CALIB_D_LIST:
        assert d % N_HEAD == 0 and (d // N_HEAD) % 2 == 0, f"d={d} invalid for rotary"
        print(f"training d={d}, m={m}, seed={CALIB_SEED} ...")
        curve = train_relay_calibration(d, m, CALIB_SEED, train_d, train_l, train_p,
                                        eval_d, eval_l, eval_p,
                                        max_steps=CALIB_MAX_STEPS, eval_every=CALIB_EVAL_EVERY)
        calibration_results[(d, m)] = curve
        verdict = classify_curve(curve)
        print(f"  d={d:>3d} m={m:>3d}: final acc={curve[-1][1]:.3f}  -> {verdict}")

print("\ncalibration grid done.")

training d=8, m=8, seed=0 ...
  d=  8 m=  8: final acc=0.983  -> SOLVED
training d=32, m=8, seed=0 ...
  d= 32 m=  8: final acc=1.000  -> SOLVED
training d=96, m=8, seed=0 ...
  d= 96 m=  8: final acc=1.000  -> SOLVED
training d=8, m=16, seed=0 ...
  d=  8 m= 16: final acc=0.471  -> FLAT AT CHANCE (genuinely stuck)
training d=32, m=16, seed=0 ...
  d= 32 m= 16: final acc=0.471  -> FLAT AT CHANCE (genuinely stuck)
training d=96, m=16, seed=0 ...


KeyboardInterrupt: 

In [ ]:
CALIB_D_LIST = [8, 32, 96]
CALIB_M_LIST = [10, 12, 14]
CALIB_SEED = 0
CALIB_MAX_STEPS = 60_000
CALIB_EVAL_EVERY = 2000
CALIB_N_TRAIN_TREES = 8_000
CALIB_N_EVAL_TREES = 1_000
CALIB_BATCH = 1024          # H200 -- these are tiny models, batch=64 was leaving the GPU idle
CALIB_LR = 1.5e-3           # halved from the 3e-3 default -- that default plausibly caused the
                            # constant-output collapse seen at m>=16 in the 60k run

calibration_results = {}
_calib_data_cache = {}

for m in CALIB_M_LIST:
    if m not in _calib_data_cache:
        train_d, train_l, train_p = make_relay_only_dataset(check_rules, CALIB_N_TRAIN_TREES,
                                                             seed=100 + m, seq_len=SEQ_LEN_CHECK, m=m)
        eval_d, eval_l, eval_p = make_relay_only_dataset(check_rules, CALIB_N_EVAL_TREES,
                                                         seed=900 + m, seq_len=SEQ_LEN_CHECK, m=m)
        _calib_data_cache[m] = (train_d, train_l, train_p, eval_d, eval_l, eval_p)
    train_d, train_l, train_p, eval_d, eval_l, eval_p = _calib_data_cache[m]

    for d in CALIB_D_LIST:
        assert d % N_HEAD == 0 and (d // N_HEAD) % 2 == 0, f"d={d} invalid for rotary"
        print(f"training d={d}, m={m}, seed={CALIB_SEED} ...")
        curve = train_relay_calibration(d, m, CALIB_SEED, train_d, train_l, train_p,
                                        eval_d, eval_l, eval_p,
                                        max_steps=CALIB_MAX_STEPS, eval_every=CALIB_EVAL_EVERY,
                                        batch=CALIB_BATCH, lr=CALIB_LR)
        calibration_results[(d, m)] = curve
        verdict = classify_curve(curve)
        last_pred_mean = curve[-1][3]
        collapsed = last_pred_mean < 0.03 or last_pred_mean > 0.97
        flag = "  *** COLLAPSED (constant output) ***" if collapsed else ""
        print(f"  d={d:>3d} m={m:>3d}: final acc={curve[-1][1]:.3f}  pred_mean={last_pred_mean:.3f}  -> {verdict}{flag}")

print("\ncalibration grid done.")

training d=8, m=10, seed=0 ...


## 12. Calibration results — table + curves

Read this the way the spec's §2 describes: look for the **qualitative
signature**, not just the endpoint. A cell that's still climbing at
`CALIB_MAX_STEPS` needs more steps before you can call it "too hard" — only
a curve that's genuinely flat near chance across steps is evidence of a real
capacity limit at that `(d, m)`.

In [ ]:
print(f"{'d':>4s} {'m':>4s} {'final_acc':>10s}  verdict")
for (d, m), curve in sorted(calibration_results.items(), key=lambda kv: (kv[0][1], kv[0][0])):
    print(f"{d:>4d} {m:>4d} {curve[-1][1]:>10.3f}  {classify_curve(curve)}")

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(CALIB_D_LIST), figsize=(5 * len(CALIB_D_LIST), 4), sharey=True)
for ax, d in zip(axes, CALIB_D_LIST):
    for m in CALIB_M_LIST:
        curve = calibration_results[(d, m)]
        steps = [s for s, _, _ in curve]
        accs = [a for _, a, _ in curve]
        ax.plot(steps, accs, marker="o", label=f"m={m}")
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
    ax.set_title(f"d={d}")
    ax.set_xlabel("training step")
    ax.set_ylim(0.4, 1.02)
    ax.legend(fontsize=8)
axes[0].set_ylabel("held-out readout accuracy")
plt.tight_layout()
plt.show()

print()
print("Pick CALIBRATED_M in the next cell: the largest m where the LOWEST d you")
print("plan to test in the real sweep still reliably solves the task within budget,")
print("while at least one d is capacity-sensitive (not saturating instantly).")

## 13. Set the calibrated `m`

**Manual step.** Inspect Section 12's table/plot, then set `CALIBRATED_M`
below. Left as `None` on purpose — the cell asserts so the notebook can't
silently proceed with an unexamined choice.

In [ ]:
CALIBRATED_M = None   # <-- SET THIS after inspecting Section 12

assert CALIBRATED_M is not None, (
    "Inspect the calibration curves above and set CALIBRATED_M before continuing. "
    "If nothing in CALIB_M_LIST looks like a clean 'boundary band' yet (e.g. every m "
    "already saturates at every d, or every m is flat everywhere), widen CALIB_M_LIST "
    "and/or raise CALIB_MAX_STEPS and rerun Section 11 before picking one."
)
print(f"using CALIBRATED_M = {CALIBRATED_M}")

## 14. Full relay + syncretism dataset/eval-set builders

Now the real 3-language construction: shared-tree pretraining mix (for
training) and a matched eval set across L_A/L_B/L_C built from the *same*
underlying trees + bits (for the probes) — same pattern as Exp 6b's
`build_matched_eval_set_eps`, extended with the relay preamble and the
readout-leaf override. Also records, per tree, the **running composed
permutation** at every preamble position (index into a canonical enumeration
of S5) — the target for the IoI-analog control probe in Section 18.

In [ ]:
import itertools
S5_ELEMENTS = list(itertools.permutations(range(5)))
S5_INDEX = {p: i for i, p in enumerate(S5_ELEMENTS)}   # 120 classes

def running_perm_indices(bits):
    # Index (0..119) of the running composed permutation after each prefix
    # g_1..g_i, for i = 1..len(bits).
    out = []
    G = IDENTITY_PERM
    for b in bits:
        G = compose(G, GEN_PERM[b])
        out.append(S5_INDEX[G])
    return out


def make_shared_relay_mix(rules, n_trees, seed, seq_len, eps_level, m, mix=MIX):
    # Packed pretraining stream (like Exp 6b's make_shared_tree_mix_eps), with
    # the relay preamble + readout-leaf override added. Rows are packed
    # end-to-end (not one-tree-per-row) since this is for LM pretraining, not
    # the probes -- probes use build_relay_matched_eval_set below instead.
    surf = {"L_A": make_surface_map(0), "L_B": make_surface_map(eps_level), "L_C": make_surface_map(eps_level)}
    rng = random.Random(seed)
    stream = []
    target_len = n_trees * (m + 30)   # rough pre-alloc hint only
    while len(stream) < target_len:
        lang = rng.choices(LANG_NAMES, weights=[mix[n] for n in LANG_NAMES])[0]
        tree = sample_tree_indexed(rules, rng)
        bits = sample_relay_chain(m, rng)
        label = f_readout(compose_chain(bits))
        stream.append(BOS[lang])
        stream.extend(relay_preamble_tokens(bits))
        for true_t, leaf_id, _ in linearize_with_paths(tree, LANGS[lang]):
            if leaf_id == 0:
                true_t = label
            stream.append(surf[lang][true_t])
    stream = torch.tensor(stream, dtype=torch.long)
    n_chunks = len(stream) // (seq_len + 1)
    return stream[: n_chunks * (seq_len + 1)].view(n_chunks, seq_len + 1)


def build_relay_matched_eval_set(rules, n_trees, seed, seq_len, eps_level, m, max_leaves=None):
    # Same underlying (tree, bits) pair reused across L_A/L_B/L_C. Returns,
    # per language: idx (surface tokens the model saw) and readout_pos
    # (position of the readout leaf in THIS language's order). Also returns
    # labels (f(G), shared across languages), preamble_pos (the m preamble
    # positions -- IDENTICAL across languages by construction), and
    # preamble_perm_idx (running composed-perm class at each preamble step,
    # also shared across languages).
    surf = {"L_A": make_surface_map(0), "L_B": make_surface_map(eps_level), "L_C": make_surface_map(eps_level)}
    rng = random.Random(seed)
    cap = seq_len - 1 - m
    kept = []
    while len(kept) < n_trees:
        tree = sample_tree_indexed(rules, rng)
        leaves = linearize_with_paths(tree, LANGS["L_A"])
        if len(leaves) > cap or (max_leaves is not None and len(leaves) > max_leaves):
            continue
        bits = sample_relay_chain(m, rng)
        kept.append((tree, bits))

    labels = torch.tensor([f_readout(compose_chain(b)) for _, b in kept], dtype=torch.long)
    perm_idx = torch.tensor([running_perm_indices(b) for _, b in kept], dtype=torch.long)   # (n_trees, m)
    preamble_pos = torch.arange(1, 1 + m, dtype=torch.long)   # identical across languages, positions 1..m

    out = {}
    for lang in LANG_NAMES:
        idx = torch.zeros(n_trees, seq_len, dtype=torch.long)
        readout_pos = torch.zeros(n_trees, dtype=torch.long)
        for t, (tree, bits) in enumerate(kept):
            seq = [BOS[lang]] + relay_preamble_tokens(bits)
            for true_t, leaf_id, _ in linearize_with_paths(tree, LANGS[lang]):
                if leaf_id == 0:
                    true_t = int(labels[t])
                    readout_pos[t] = len(seq)
                seq.append(surf[lang][true_t])
            idx[t, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        out[lang] = dict(idx=idx, readout_pos=readout_pos)
    return dict(per_lang=out, labels=labels, preamble_pos=preamble_pos, preamble_perm_idx=perm_idx)


# smoke test: preamble positions/labels must be identical across languages by construction
_es = build_relay_matched_eval_set(check_rules, 20, seed=5, seq_len=SEQ_LEN_CHECK, eps_level=1, m=CALIBRATED_M)
for lang in LANG_NAMES:
    idx = _es["per_lang"][lang]["idx"]
    pre = idx[:, 1:1 + CALIBRATED_M]
    print(f"{lang}: preamble block shape {pre.shape}")
_all_pre = [tuple(_es["per_lang"][lang]["idx"][0, 1:1+CALIBRATED_M].tolist()) for lang in LANG_NAMES]
assert len(set(_all_pre)) == 1, "preamble differs across languages in the real eval-set builder -- bug"
print("preamble identical across languages in the real (multilingual) builder: OK")

## 15. Train ONE pilot checkpoint

Single `(d, seed)`, single `eps_level`, at `CALIBRATED_M` — reuses the same
`train_worker.py` subprocess infra the real sweep will use, so this is a
direct dry run of that pipeline on one point rather than a separate code
path. **Pick `PILOT_D` from the high end of Section 12's capacity-sensitive
band** — the point most likely to show a real gap if one exists.

In [ ]:
PILOT_D = 96          # <-- adjust based on calibration results
PILOT_SEED = 0
PILOT_EPS_LEVEL = 1
PILOT_STEPS = 90_000
PILOT_N_TREES = 50_000

CK_PILOT = "./exp6b_v2_pilot"; os.makedirs(CK_PILOT, exist_ok=True)
pilot_data_path = os.path.join(CK_PILOT, f"data_eps{PILOT_EPS_LEVEL}_m{CALIBRATED_M}_seed{PILOT_SEED}.pt")
pilot_ckpt_path = os.path.join(CK_PILOT, f"shared_eps{PILOT_EPS_LEVEL}_m{CALIBRATED_M}_d{PILOT_D}_seed{PILOT_SEED}.pt")

if not os.path.exists(pilot_data_path):
    _data = make_shared_relay_mix(check_rules, n_trees=PILOT_N_TREES, seed=PILOT_SEED,
                                  seq_len=SEQ_LEN_CHECK, eps_level=PILOT_EPS_LEVEL, m=CALIBRATED_M)
    torch.save(_data.cpu(), pilot_data_path)
    print(f"saved {pilot_data_path}  shape={_data.shape}")

In [ ]:
%%writefile train_worker.py
import argparse, os, time
import torch
import torch.nn as nn
import torch.nn.functional as F

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert (n_embd // n_head) % 2 == 0, "head_dim must be EVEN for rotary embeddings"
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--d", type=int, required=True)
    ap.add_argument("--seed", type=int, required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--vocab", type=int, required=True)
    ap.add_argument("--seq_len", type=int, required=True)
    ap.add_argument("--steps", type=int, default=90000)
    ap.add_argument("--n_head", type=int, default=4)
    ap.add_argument("--n_layer", type=int, default=12)
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--lr", type=float, default=3e-3)
    ap.add_argument("--compile", type=int, default=0)
    a = ap.parse_args()

    torch.set_num_threads(1)
    device = torch.device(a.device)
    torch.manual_seed(a.seed)
    data = torch.load(a.data).to(device)
    model = TinyRotaryGPT(a.vocab, n_embd=a.d, n_layer=a.n_layer, n_head=a.n_head,
                          max_seq_len=a.seq_len + 1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, fused=True)
    fwd = torch.compile(model) if a.compile else model
    n = data.shape[0]
    buf = torch.zeros(a.steps, device=device)
    t0 = time.time()
    for step in range(a.steps):
        idx = torch.randint(0, n, (a.batch,), device=device)
        batch = data[idx]
        x, y = batch[:, :-1], batch[:, 1:]
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            _, loss = fwd(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
        buf[step] = loss.detach().float()
        if (step + 1) % 5000 == 0:
            print(f"step {step+1:>7d}: mean loss over last 5000 = {buf[step-4999:step+1].mean().item():.4f}"
                  f"  ({(step + 1) / (time.time() - t0):.0f} it/s)", flush=True)
        if (step + 1) % 10000 == 0 or step == a.steps - 1:
            tmp = a.ckpt + ".tmp"
            torch.save({"model_state": model.state_dict(), "opt_state": opt.state_dict(),
                        "step": step + 1, "losses": buf[:step + 1].tolist()}, tmp)
            os.replace(tmp, a.ckpt)

if __name__ == "__main__":
    main()

In [ ]:
import subprocess, sys, time

def ckpt_step(path):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location="cpu", weights_only=False)["step"]

def run_parallel_jobs(jobs, vocab_size, seq_len, max_par=MAX_PAR, device=DEVICE):
    jobs = [j for j in jobs if ckpt_step(j["ckpt_path"]) != j["steps"]]
    print(f"{len(jobs)} runs to do, {max_par} at a time")

    def launch(j):
        log = open(j["ckpt_path"] + ".log", "w")
        cmd = [sys.executable, "train_worker.py", "--d", str(j["d"]), "--seed", str(j["seed"]),
               "--data", j["data_path"], "--ckpt", j["ckpt_path"], "--vocab", str(vocab_size),
               "--seq_len", str(seq_len), "--steps", str(j["steps"]), "--n_head", str(j["n_head"]),
               "--device", str(device)]
        return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT), j, log

    running = []
    while jobs or running:
        still = []
        for p, j, log in running:
            rc = p.poll()
            if rc is None:
                still.append((p, j, log))
            else:
                log.close()
                tag = f"d={j['d']} seed={j['seed']} ({os.path.basename(j['ckpt_path'])})"
                print(f"finished {tag}  exit code {rc}" + ("" if rc == 0 else "   <-- check the .log file"))
        running = still
        while jobs and len(running) < max_par:
            running.append(launch(jobs.pop(0)))
        time.sleep(10)
    print("all queued jobs finished (check exit codes above for failures)")

def assert_all_complete(paths_and_steps):
    missing = [(p, s, ckpt_step(p)) for p, s in paths_and_steps if ckpt_step(p) != s]
    assert not missing, f"incomplete runs (path, target_step, actual_step): {missing}"
    print(f"all {len(paths_and_steps)} runs complete")

pilot_job = [dict(d=PILOT_D, seed=PILOT_SEED, n_head=N_HEAD, steps=PILOT_STEPS,
                  data_path=pilot_data_path, ckpt_path=pilot_ckpt_path)]
run_parallel_jobs(pilot_job, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN_CHECK, max_par=1)
assert_all_complete([(pilot_ckpt_path, PILOT_STEPS)])

## 16. Probe infrastructure (reused from Exp 6b)

In [ ]:
def load_shared_ckpt(path, d_model, seq_len, vocab_size=VOCAB_SIZE, n_layer=12, n_head=N_HEAD, device=DEVICE):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    m = TinyRotaryGPT(vocab_size, n_embd=d_model, n_layer=n_layer, n_head=n_head,
                      max_seq_len=seq_len + 1).to(device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()
    tail = ckpt["losses"][-1000:]
    print(f"{path}: step={ckpt['step']}, tail-1000 mean loss={sum(tail)/len(tail):.4f}")
    return m


@torch.no_grad()
def extract_hidden_states(model, idx_tensor, layers, batch_size=32, device=DEVICE):
    model.eval()
    activations = {l: [] for l in layers}

    def make_hook(l):
        def hook(module, inp, out):
            activations[l].append(out.detach().float().cpu())
        return hook

    handles = [model.blocks[l].register_forward_hook(make_hook(l)) for l in layers]
    try:
        for i in range(0, idx_tensor.shape[0], batch_size):
            batch = idx_tensor[i:i + batch_size].to(device)
            with autocast_ctx():
                _ = model(batch)
    finally:
        for h in handles:
            h.remove()
    return {l: torch.cat(activations[l], dim=0) for l in layers}


def split_trees(n_trees, test_frac=0.2, seed=0):
    import numpy as np
    rng = np.random.RandomState(seed)
    idx = rng.permutation(n_trees)
    n_test = int(n_trees * test_frac)
    return set(idx[n_test:].tolist()), set(idx[:n_test].tolist())


def _fit_logreg(X, y, n_class, epochs=300, lr=1e-2, device=DEVICE):
    mu, sd = X.mean(0), X.std(0) + 1e-6
    Xs = ((X - mu) / sd).to(device); ys = y.to(device)
    W = nn.Linear(X.shape[1], n_class).to(device)
    opt = torch.optim.AdamW(W.parameters(), lr=lr)
    for _ in range(epochs):
        loss = F.cross_entropy(W(Xs), ys); opt.zero_grad(); loss.backward(); opt.step()
    return W, mu, sd


@torch.no_grad()
def _score(W, mu, sd, X, y, device=DEVICE):
    return (W(((X - mu) / sd).to(device)).argmax(-1).cpu() == y).float().mean().item()

## 17. Probe 1 (primary) — `f(G)` recovery at the readout leaf

**Interpretation gate (spec §4): check L_A's own recovery first.** L_A's
readout is not syncretism-collapsed, so a linear probe on its hidden states
should sit near ceiling — if it doesn't, the model hasn't even learned the
base relay task at this `d`/step budget and a B/C gap here is uninterpretable
(this is exactly the trap the §2 calibration gate exists to prevent, so a
failure here after passing calibration would itself be a red flag worth
chasing down before going further).

L_B/L_C readout tokens ARE syncretism-collapsed (`eps_level=PILOT_EPS_LEVEL`)
— this is the actual ε-floor test.

In [ ]:
PILOT_EVAL_TREES = 2000
pilot_eval_seed = 999

pilot_eval = build_relay_matched_eval_set(check_rules, PILOT_EVAL_TREES, seed=pilot_eval_seed,
                                          seq_len=SEQ_LEN_CHECK, eps_level=PILOT_EPS_LEVEL, m=CALIBRATED_M)
pilot_model = load_shared_ckpt(pilot_ckpt_path, PILOT_D, SEQ_LEN_CHECK)
train_ids, test_ids = split_trees(PILOT_EVAL_TREES, test_frac=0.2, seed=0)

PROBE_LAYERS = (6, 8, 10, 11)

def readout_probe(lang, layers=PROBE_LAYERS):
    idx = pilot_eval["per_lang"][lang]["idx"]
    pos = pilot_eval["per_lang"][lang]["readout_pos"]
    labels = pilot_eval["labels"]
    H = extract_hidden_states(pilot_model, idx, layers)
    results = {}
    for L in layers:
        h = H[L]
        X = torch.stack([h[t, pos[t] - 1] for t in range(idx.shape[0])])   # position BEFORE the readout token predicts it
        train_idx = torch.tensor(sorted(train_ids)); test_idx = torch.tensor(sorted(test_ids))
        W, mu, sd = _fit_logreg(X[train_idx], labels[train_idx], n_class=2)
        acc = _score(W, mu, sd, X[test_idx], labels[test_idx])
        results[L] = acc
    return results

print("=== Probe 1: f(G) recovery at the readout leaf ===\n")
print("L_A (unambiguous -- interpretation gate, check THIS first):")
acc_A = readout_probe("L_A")
for L, a in acc_A.items():
    print(f"  layer {L}: {a:.3f}")

if min(acc_A.values()) < 0.85:
    print("\n*** L_A itself isn't near ceiling -- do NOT interpret B/C below yet. Either PILOT_D is still "
          "in the capacity-limited range for the base relay task, or something in the pipeline broke; "
          "go back to Section 12's calibration curve for this d before trusting anything past this point. ***\n")

print("\nL_B (syncretized):")
acc_B = readout_probe("L_B")
for L, a in acc_B.items():
    print(f"  layer {L}: {a:.3f}")

print("\nL_C (syncretized):")
acc_C = readout_probe("L_C")
for L, a in acc_C.items():
    print(f"  layer {L}: {a:.3f}")

## 18. Probe 2 (control) — IoI-analog: recover the *visible* relay state

Sanity check on the instrument itself, not a claim about ε. The preamble is
byte-identical across L_A/L_B/L_C by construction (Section 8), so a probe
targeting the **running composed permutation** at a preamble position should
show **no cross-language gap** at any layer. If this control also shows a
growing L_A-vs-L_B/L_C gap, that's evidence the probe/CE-gap instrument is
manufacturing separation out of nothing, and Probe 1's result above should
not be trusted until this is resolved — check this BEFORE reading anything
into Probe 1's B/C gap.

Checked at two preamble positions: early (`i=0`, first generator, low
information) and late (`i=m-1`, full running state, most information).

In [ ]:
def control_probe(lang, preamble_step, layers=PROBE_LAYERS):
    # preamble_step in [0, m-1]. Target: 120-way running-permutation class
    # after prefix g_1..g_{preamble_step+1}, shared across languages.
    idx = pilot_eval["per_lang"][lang]["idx"]
    pos = int(pilot_eval["preamble_pos"][preamble_step])          # sequence position of this preamble token
    y = pilot_eval["preamble_perm_idx"][:, preamble_step]
    H = extract_hidden_states(pilot_model, idx, layers)
    results = {}
    for L in layers:
        h = H[L]
        X = h[:, pos]      # hidden state AT the preamble token position (identical position across languages)
        train_idx = torch.tensor(sorted(train_ids)); test_idx = torch.tensor(sorted(test_ids))
        # 120-way classification with a few hundred eval trees is data-starved by design here --
        # this control only needs to show NO CROSS-LANGUAGE GAP, not high absolute accuracy.
        W, mu, sd = _fit_logreg(X[train_idx], y[train_idx], n_class=len(S5_ELEMENTS), epochs=500)
        acc = _score(W, mu, sd, X[test_idx], y[test_idx])
        results[L] = acc
    return results

print("=== Probe 2 (control): running composed-permutation recovery ===\n")
for step_name, step_idx in [("early (i=0)", 0), ("late (i=m-1)", CALIBRATED_M - 1)]:
    print(f"-- preamble step {step_name} --")
    accs = {lang: control_probe(lang, step_idx) for lang in LANG_NAMES}
    for L in PROBE_LAYERS:
        row = "  ".join(f"{lang}={accs[lang][L]:.3f}" for lang in LANG_NAMES)
        print(f"  layer {L}: {row}")
    gaps = [max(accs[lang][L] for lang in LANG_NAMES) - min(accs[lang][L] for lang in LANG_NAMES)
            for L in PROBE_LAYERS]
    print(f"  max cross-language gap across these layers: {max(gaps):.3f}")
    if max(gaps) > 0.15:
        print("  *** WARNING: control probe shows a real cross-language gap on an identical-by-construction "
              "preamble -- treat Probe 1's B/C result as suspect until this is understood. ***")
    print()

## 19. Wrap-up — what this notebook did and did not check

**Read Section 17 with Section 18's result in hand, not before**, per the
control-first logic above.

Green light for the real sweep looks like: L_A's Probe-1 recovery near
ceiling, a real L_B/L_C gap below it (ideally `C > B`, matching the
word-order-distance story from Exp 6, though the spec explicitly doesn't
require this ordering), and Probe 2 flat across languages at both preamble
steps.

**Not built here, still pending before the real sweep (spec §3/§4):**
- the disjoint-grammar control carrying the *same* relay burden (same `m`,
  same generator vocab, same preamble placement) — required so the control
  doesn't reintroduce the exact task-difficulty-mismatch confound already
  caught once in Experiment 6, now with a much larger potential lever;
- the full `eps_level × d × seed` grid;
- budget-matched dedicated models and the A′ calibration condition.

Also not re-run here (deliberately, per spec §5): the old §3-style
sliding-window ambiguity pre-flight check. It would trivially pass for any
relay-chain mechanism regardless of genuine shortcut-resistance, since
attention isn't limited to short windows -- it proves nothing about this
design. The count-bucket audit (Section 6) and the calibration gate
(Sections 9-13) are the correct diagnostics here, not that check.